<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data/processed')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 50
NOISE_STD = 0.05

In [3]:
class CNNLSTMHybrid(nn.Module):
    def __init__(self, in_channels, num_classes=6, noise_std=0.0):
        super(CNNLSTMHybrid, self).__init__()
        self.noise_std = noise_std

        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64)
        )

        self.lstm = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            x = x + noise

        out = self.cnn(x)
        out = out.permute(0, 2, 1)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

In [4]:
def prepare_dataloaders(features_path, labels_path):
    if not os.path.exists(features_path) or not os.path.exists(labels_path):
        return None, None, None, 0

    features_tensor = torch.load(features_path)
    labels_tensor = torch.load(labels_path).long()

    dataset_size = len(features_tensor)
    if dataset_size == 0:
        return None, None, None, 0

    train_size = int(0.7 * dataset_size)
    val_size = int(0.15 * dataset_size)
    test_size = dataset_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        TensorDataset(features_tensor, labels_tensor),
        [train_size, val_size, test_size]
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, test_loader, dataset_size

In [5]:
def train_and_evaluate_model(device_name, in_channels, features_name, labels_name, save_name):
    print(f"\n[{device_name} 모델 학습 준비]")

    features_path = os.path.join(PROCESSED_DIR, features_name)
    labels_path = os.path.join(PROCESSED_DIR, labels_name)

    train_loader, val_loader, test_loader, dataset_size = prepare_dataloaders(features_path, labels_path)

    if train_loader is None or dataset_size == 0:
        print(f"{device_name} 데이터를 찾을 수 없거나 데이터가 부족하여 학습을 건너뜁니다.")
        return

    print(f"데이터셋 분할 완료 - 총 {dataset_size}개 (학습: {len(train_loader.dataset)}개, 검증: {len(val_loader.dataset)}개, 테스트: {len(test_loader.dataset)}개)")

    model = CNNLSTMHybrid(in_channels=in_channels, num_classes=6, noise_std=NOISE_STD)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_accuracy = 0.0
    best_model_path = os.path.join(CHECKPOINT_DIR, save_name)

    print(f"{device_name} 모델 학습을 시작합니다...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        if total > 0:
            accuracy = 100 * correct / total
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {accuracy:.2f} 퍼센트")

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                torch.save(model.state_dict(), best_model_path)

    print(f"{device_name} 학습 및 검증 완료. 최고 검증 정확도: {best_accuracy:.2f} 퍼센트")
    print(f"저장된 최고 성능 가중치로 최종 테스트 평가를 진행합니다...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    if test_total > 0:
        test_accuracy = 100 * test_correct / test_total
        print(f"[{device_name} 최종 테스트 정확도: {test_accuracy:.2f} 퍼센트]")
        print(f"최고 성능 가중치가 {best_model_path}에 저장되었습니다.\n")
    else:
        print("테스트 데이터가 부족하여 평가를 생략합니다.\n")

In [6]:
train_and_evaluate_model(
    device_name='스마트폰',
    in_channels=6,
    features_name='smartphone_features.pt',
    labels_name='smartphone_labels.pt',
    save_name='smartphone_best_model.pth'
)

train_and_evaluate_model(
    device_name='에스펜',
    in_channels=2,
    features_name='spen_features.pt',
    labels_name='spen_labels.pt',
    save_name='spen_best_model.pth'
)


[스마트폰 모델 학습 준비]
데이터셋 분할 완료 - 총 3132개 (학습: 2192개, 검증: 469개, 테스트: 471개)
스마트폰 모델 학습을 시작합니다...
Epoch 1/50, Loss: 0.9931, Val Accuracy: 68.23 퍼센트
Epoch 10/50, Loss: 0.3049, Val Accuracy: 87.85 퍼센트
Epoch 20/50, Loss: 0.1465, Val Accuracy: 91.68 퍼센트
Epoch 30/50, Loss: 0.1102, Val Accuracy: 92.11 퍼센트
Epoch 40/50, Loss: 0.0791, Val Accuracy: 92.54 퍼센트
Epoch 50/50, Loss: 0.0746, Val Accuracy: 92.96 퍼센트
스마트폰 학습 및 검증 완료. 최고 검증 정확도: 94.46 퍼센트
저장된 최고 성능 가중치로 최종 테스트 평가를 진행합니다...
[스마트폰 최종 테스트 정확도: 93.21 퍼센트]
최고 성능 가중치가 /content/drive/MyDrive/point-6/ai_model/checkpoints/smartphone_best_model.pth에 저장되었습니다.


[에스펜 모델 학습 준비]
데이터셋 분할 완료 - 총 1941개 (학습: 1358개, 검증: 291개, 테스트: 292개)
에스펜 모델 학습을 시작합니다...
Epoch 1/50, Loss: 1.1042, Val Accuracy: 85.22 퍼센트
Epoch 10/50, Loss: 0.0355, Val Accuracy: 95.88 퍼센트
Epoch 20/50, Loss: 0.0282, Val Accuracy: 96.22 퍼센트
Epoch 30/50, Loss: 0.0031, Val Accuracy: 98.28 퍼센트
Epoch 40/50, Loss: 0.0043, Val Accuracy: 96.91 퍼센트
Epoch 50/50, Loss: 0.0038, Val Accuracy: 97.59 퍼센트
에스펜 학습